Libaries/Tools

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix, classification_report
import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score



c:\Users\mikko\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LOAD DATASET

In [2]:
#dataset
matches = pd.read_csv('C:/Users/mikko/OneDrive/python_projektit/ML_dec_tree_football/data_csv/Matches.csv')

C:\Users\mikko\AppData\Local\Temp\ipykernel_24156\583763451.py:2: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  matches = pd.read_csv('C:/Users/mikko/OneDrive/python_projektit/ML_dec_tree_football/data_csv/Matches.csv')


DROP COLUMNS AND FILL_EMPTY

In [3]:
matches_cleaned = matches.drop(columns=['MatchTime', 'HomeFouls', 'AwayFouls',
    'HomeYellow', 'AwayYellow', 'HomeRed', 'AwayRed',
    'MaxHome', 'MaxDraw', 'MaxAway',
    'Over25', 'Under25', 'MaxOver25', 'MaxUnder25',
    'HandiSize', 'HandiHome', 'HandiAway',
    'C_LTH', 'C_LTA', 'C_VHD', 'C_VAD',
    'C_HTB', 'C_PHB'
])

matches_cleaned = matches_cleaned.dropna()

FILTER AND REPLACE VALUES

In [4]:
matches_cleaned = matches_cleaned[matches_cleaned['Division'] == 'E0']
matches_cleaned['MatchDate'] = pd.to_datetime(matches_cleaned['MatchDate'])

matches_cleaned[['HomeTeam', 'AwayTeam']] = (
    matches_cleaned[['HomeTeam', 'AwayTeam']]
        .replace({
            "Nott'm Forest": "Nottingham Forest",
            "Nottm Forest": "Nottingham Forest"
        })
)

matches_cleaned['HomePoints'] = matches_cleaned['FTResult'].map({
    'H': 3,
    'D': 1,
    'A': 0
})

matches_cleaned['AwayPoints'] = matches_cleaned['FTResult'].map({
    'H': 0,
    'D': 1,
    'A': 3
})

matches_cleaned['FTResult'] = matches_cleaned['FTResult'].map({
    'H': 0,
    'D': 1,
    'A': 2
})

In [5]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

matches_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9325 entries, 154 to 230510
Data columns (total 27 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Division     9325 non-null   object        
 1   MatchDate    9325 non-null   datetime64[ns]
 2   HomeTeam     9325 non-null   object        
 3   AwayTeam     9325 non-null   object        
 4   HomeElo      9325 non-null   float64       
 5   AwayElo      9325 non-null   float64       
 6   Form3Home    9325 non-null   float64       
 7   Form5Home    9325 non-null   float64       
 8   Form3Away    9325 non-null   float64       
 9   Form5Away    9325 non-null   float64       
 10  FTHome       9325 non-null   float64       
 11  FTAway       9325 non-null   float64       
 12  FTResult     9325 non-null   int64         
 13  HTHome       9325 non-null   float64       
 14  HTAway       9325 non-null   float64       
 15  HTResult     9325 non-null   object        
 16  HomeSho

In [6]:
matches_cleaned['margin'] = 1 / matches_cleaned['OddHome'] + 1 / matches_cleaned['OddDraw'] + 1 / matches_cleaned['OddAway']
matches_cleaned['HomeOdds'] = 1 / matches_cleaned['OddHome'] / matches_cleaned['margin']
matches_cleaned['DrawOdds'] = 1 / matches_cleaned['OddDraw'] / matches_cleaned['margin']
matches_cleaned['AwayOdds'] = 1 / matches_cleaned['OddAway'] / matches_cleaned['margin']


In [7]:
matches_cleaned = matches_cleaned.drop(columns=['OddHome', 'OddDraw', 'OddAway', 'margin'])

In [8]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

matches_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9325 entries, 154 to 230510
Data columns (total 27 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Division     9325 non-null   object        
 1   MatchDate    9325 non-null   datetime64[ns]
 2   HomeTeam     9325 non-null   object        
 3   AwayTeam     9325 non-null   object        
 4   HomeElo      9325 non-null   float64       
 5   AwayElo      9325 non-null   float64       
 6   Form3Home    9325 non-null   float64       
 7   Form5Home    9325 non-null   float64       
 8   Form3Away    9325 non-null   float64       
 9   Form5Away    9325 non-null   float64       
 10  FTHome       9325 non-null   float64       
 11  FTAway       9325 non-null   float64       
 12  FTResult     9325 non-null   int64         
 13  HTHome       9325 non-null   float64       
 14  HTAway       9325 non-null   float64       
 15  HTResult     9325 non-null   object        
 16  HomeSho

ROLLING AVERAGES FOR TOTAL POINTS

In [9]:
matches_cleaned['MatchDate'] = pd.to_datetime(matches_cleaned['MatchDate'])

# Sort chronologically and create stable match id
matches_cleaned = matches_cleaned.sort_values('MatchDate').reset_index(drop=True)
matches_cleaned['MatchID'] = matches_cleaned.index

# --- Long format: one row per (match, team) with points earned ---
home_long = matches_cleaned[['MatchID', 'MatchDate', 'HomeTeam', 'HomePoints']].copy()
home_long = home_long.rename(columns={'HomeTeam': 'Team', 'HomePoints': 'Points'})
home_long['Side'] = 'Home'

away_long = matches_cleaned[['MatchID', 'MatchDate', 'AwayTeam', 'AwayPoints']].copy()
away_long = away_long.rename(columns={'AwayTeam': 'Team', 'AwayPoints': 'Points'})
away_long['Side'] = 'Away'

teams_long = pd.concat([home_long, away_long], ignore_index=True)

# --- Rolling average points over last 20 games per team (BEFORE match) ---
teams_long = teams_long.sort_values(['Team', 'MatchDate', 'MatchID']).reset_index(drop=True)

g = teams_long.groupby('Team')['Points']

# rolling avg of previous 20 matches: shift to exclude current match
teams_long['RollingAvgPoints20Before'] = (
    g.apply(lambda s: s.shift(1).rolling(window=20, min_periods=1).mean())
     .reset_index(level=0, drop=True)
)

# --- Merge back to match table as Home/Away features ---
home_feat = teams_long[teams_long['Side'] == 'Home'][['MatchID', 'RollingAvgPoints20Before']].rename(
    columns={'RollingAvgPoints20Before': 'HomeTotalPointsRoll20'}  # name kept, now rolling avg
)
away_feat = teams_long[teams_long['Side'] == 'Away'][['MatchID', 'RollingAvgPoints20Before']].rename(
    columns={'RollingAvgPoints20Before': 'AwayTotalPointsRoll20'}  # name kept, now rolling avg
)

matches_cleaned = matches_cleaned.merge(home_feat, on='MatchID', how='left')
matches_cleaned = matches_cleaned.merge(away_feat, on='MatchID', how='left')

ROLLING AVERAGES FOR HOME POINTS

In [10]:
# Ensure chronological order
matches_cleaned['MatchDate'] = pd.to_datetime(matches_cleaned['MatchDate'])
matches_cleaned = matches_cleaned.sort_values('MatchDate').reset_index(drop=True)

# --- Rolling avg of last 20 HOME matches for each team ---
matches_cleaned['HomeTeamHomePointsRolling20'] = (
    matches_cleaned
    .groupby('HomeTeam')['HomePoints']
    .apply(lambda s: s.shift(1).rolling(window=20, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

ROLLING AVERAGES FOR AWAY POINTS

In [11]:
matches_cleaned['AwayTeamAwayPointsRolling20'] = (
    matches_cleaned
    .groupby('AwayTeam')['AwayPoints']
    .apply(lambda s: s.shift(1).rolling(window=20, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

ROLLING AVERAGES FOR TOTAL SHOTS

In [12]:
# Ensure shots are numeric
matches_cleaned['HomeShots'] = pd.to_numeric(matches_cleaned['HomeShots'], errors='coerce')
matches_cleaned['AwayShots'] = pd.to_numeric(matches_cleaned['AwayShots'], errors='coerce')

# --- Long format: one row per (match, team) with shots taken in that match ---
home_long = matches_cleaned[['MatchID', 'MatchDate', 'HomeTeam', 'HomeShots']].copy()
home_long = home_long.rename(columns={'HomeTeam': 'Team', 'HomeShots': 'Shots'})
home_long['Side'] = 'Home'

away_long = matches_cleaned[['MatchID', 'MatchDate', 'AwayTeam', 'AwayShots']].copy()
away_long = away_long.rename(columns={'AwayTeam': 'Team', 'AwayShots': 'Shots'})
away_long['Side'] = 'Away'

teams_long = pd.concat([home_long, away_long], ignore_index=True)

# --- Rolling AVG shots per team over last 20 games (BEFORE match), no seasonality ---
teams_long = teams_long.sort_values(['Team', 'MatchDate', 'MatchID']).reset_index(drop=True)

g = teams_long.groupby('Team')['Shots']

teams_long['ShotsRolling20Before'] = (
    g.apply(lambda s: s.shift(1).rolling(window=20, min_periods=1).mean())
     .reset_index(level=0, drop=True)
)

# --- Merge back to match table as Home/Away features ---
home_feat = teams_long[teams_long['Side'] == 'Home'][['MatchID', 'ShotsRolling20Before']].rename(
    columns={'ShotsRolling20Before': 'HomeTeamShotsRolling20'}
)
away_feat = teams_long[teams_long['Side'] == 'Away'][['MatchID', 'ShotsRolling20Before']].rename(
    columns={'ShotsRolling20Before': 'AwayTeamShotsRolling20'}
)

matches_cleaned = matches_cleaned.merge(home_feat, on='MatchID', how='left')
matches_cleaned = matches_cleaned.merge(away_feat, on='MatchID', how='left')
# Optional cleanup:
# matches_cleaned = matches_cleaned.drop(columns=['MatchID'])

ROLLING AVERAGE FOR HOME AND AWAY SHOTS

In [13]:
# Ensure datetime + chronological order
matches_cleaned['MatchDate'] = pd.to_datetime(matches_cleaned['MatchDate'])
matches_cleaned = matches_cleaned.sort_values('MatchDate').reset_index(drop=True)

# Shots numeric
matches_cleaned['HomeShots'] = pd.to_numeric(matches_cleaned['HomeShots'], errors='coerce')
matches_cleaned['AwayShots'] = pd.to_numeric(matches_cleaned['AwayShots'], errors='coerce')

# --- Home team: rolling avg of last 20 HOME matches (exclude current) ---
matches_cleaned['HomeTeamShotsHomeRolling20'] = (
    matches_cleaned
    .groupby('HomeTeam')['HomeShots']
    .apply(lambda s: s.shift(1).rolling(window=20, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

# --- Away team: rolling avg of last 20 AWAY matches (exclude current) ---
matches_cleaned['AwayTeamShotsAwayRolling20'] = (
    matches_cleaned
    .groupby('AwayTeam')['AwayShots']
    .apply(lambda s: s.shift(1).rolling(window=20, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

# Optional: fill early NaNs (teams with no prior home/away games)
# matches_cleaned[['HomeTeamShotsHomeRolling20', 'AwayTeamShotsAwayRolling20']] = (
#     matches_cleaned[['HomeTeamShotsHomeRolling20', 'AwayTeamShotsAwayRolling20']].fillna(0)
# )

ROLLING AVERAGE FOR SHOTS ON TARGET

In [14]:
# Ensure numeric
matches_cleaned['HomeTarget'] = pd.to_numeric(matches_cleaned['HomeTarget'], errors='coerce')
matches_cleaned['AwayTarget'] = pd.to_numeric(matches_cleaned['AwayTarget'], errors='coerce')

# Make sure sorted and MatchID exists
matches_cleaned['MatchDate'] = pd.to_datetime(matches_cleaned['MatchDate'])
matches_cleaned = matches_cleaned.sort_values('MatchDate').reset_index(drop=True)
if 'MatchID' not in matches_cleaned.columns:
    matches_cleaned['MatchID'] = matches_cleaned.index

# --- Long format ---
home_long = matches_cleaned[['MatchID','MatchDate','HomeTeam','HomeTarget']].copy()
home_long = home_long.rename(columns={'HomeTeam':'Team','HomeTarget':'TargetShots'})
home_long['Side'] = 'Home'

away_long = matches_cleaned[['MatchID','MatchDate','AwayTeam','AwayTarget']].copy()
away_long = away_long.rename(columns={'AwayTeam':'Team','AwayTarget':'TargetShots'})
away_long['Side'] = 'Away'

teams_long = pd.concat([home_long, away_long], ignore_index=True)

# --- Rolling AVG per team over last 20 matches (BEFORE match), no seasonality ---
teams_long = teams_long.sort_values(['Team','MatchDate','MatchID']).reset_index(drop=True)

g = teams_long.groupby('Team')['TargetShots']

teams_long['TargetShotsRolling20Before'] = (
    g.apply(lambda s: s.shift(1).rolling(window=20, min_periods=1).mean())
     .reset_index(level=0, drop=True)
)

# --- Merge back ---
home_feat = teams_long[teams_long['Side']=='Home'][['MatchID','TargetShotsRolling20Before']] \
    .rename(columns={'TargetShotsRolling20Before':'HomeTeamTargetShotsRolling20'})

away_feat = teams_long[teams_long['Side']=='Away'][['MatchID','TargetShotsRolling20Before']] \
    .rename(columns={'TargetShotsRolling20Before':'AwayTeamTargetShotsRolling20'})

matches_cleaned = matches_cleaned.merge(home_feat, on='MatchID', how='left')
matches_cleaned = matches_cleaned.merge(away_feat, on='MatchID', how='left')

ROLLING AVERAGE TOTAL GOALS HOME AND AWAY TEAM

In [15]:
# Ensure numeric
matches_cleaned['FTHome'] = pd.to_numeric(matches_cleaned['FTHome'], errors='coerce')
matches_cleaned['FTAway'] = pd.to_numeric(matches_cleaned['FTAway'], errors='coerce')

# --- Long format: goals scored by each team in each match ---
home_long = matches_cleaned[['MatchID', 'MatchDate', 'HomeTeam', 'FTHome']].copy()
home_long = home_long.rename(columns={'HomeTeam': 'Team', 'FTHome': 'Goals'})
home_long['Side'] = 'Home'

away_long = matches_cleaned[['MatchID', 'MatchDate', 'AwayTeam', 'FTAway']].copy()
away_long = away_long.rename(columns={'AwayTeam': 'Team', 'FTAway': 'Goals'})
away_long['Side'] = 'Away'

teams_long = pd.concat([home_long, away_long], ignore_index=True)

# --- Rolling AVG goals per team over last 20 matches (BEFORE match), no seasonality ---
teams_long = teams_long.sort_values(['Team', 'MatchDate', 'MatchID']).reset_index(drop=True)

g = teams_long.groupby('Team')['Goals']

teams_long['GoalsRolling20Before'] = (
    g.apply(lambda s: s.shift(1).rolling(window=20, min_periods=1).mean())
     .reset_index(level=0, drop=True)
)

# --- Merge back to match table ---
home_feat = teams_long[teams_long['Side'] == 'Home'][['MatchID', 'GoalsRolling20Before']].rename(
    columns={'GoalsRolling20Before': 'HomeTotalGoalsRolling20'}
)
away_feat = teams_long[teams_long['Side'] == 'Away'][['MatchID', 'GoalsRolling20Before']].rename(
    columns={'GoalsRolling20Before': 'AwayTotalGoalsRolling20'}
)

matches_cleaned = matches_cleaned.merge(home_feat, on='MatchID', how='left')
matches_cleaned = matches_cleaned.merge(away_feat, on='MatchID', how='left')

ROLLING AVERAGES GOALS HOME AND AWAY

In [16]:
# Ensure numeric
matches_cleaned['FTHome'] = pd.to_numeric(matches_cleaned['FTHome'], errors='coerce')
matches_cleaned['FTAway'] = pd.to_numeric(matches_cleaned['FTAway'], errors='coerce')

# --- Home team: rolling avg goals in last 20 HOME matches (exclude current) ---
matches_cleaned['HomeGoalsHomeRolling20'] = (
    matches_cleaned
    .groupby('HomeTeam')['FTHome']
    .apply(lambda s: s.shift(1).rolling(window=20, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

# --- Away team: rolling avg goals in last 20 AWAY matches (exclude current) ---
matches_cleaned['AwayGoalsAwayRolling20'] = (
    matches_cleaned
    .groupby('AwayTeam')['FTAway']
    .apply(lambda s: s.shift(1).rolling(window=20, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

# Optional: fill early NaNs
# matches_cleaned[['HomeGoalsHomeRolling20','AwayGoalsAwayRolling20']] = (
#     matches_cleaned[['HomeGoalsHomeRolling20','AwayGoalsAwayRolling20']].fillna(0)
# )

ROLLING AVERAGE GOALS CONCEDED HOME AND AWAY FOR HOME AND AWAY TEAM

In [17]:
# Ensure numeric
matches_cleaned['FTHome'] = pd.to_numeric(matches_cleaned['FTHome'], errors='coerce')
matches_cleaned['FTAway'] = pd.to_numeric(matches_cleaned['FTAway'], errors='coerce')

# --- Home team conceded at home: rolling avg conceded in last 20 HOME matches (exclude current) ---
matches_cleaned['HomeConcededHomeRolling20'] = (
    matches_cleaned
    .groupby('HomeTeam')['FTAway']
    .apply(lambda s: s.shift(1).rolling(window=20, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

# --- Away team conceded away: rolling avg conceded in last 20 AWAY matches (exclude current) ---
matches_cleaned['AwayConcededAwayRolling20'] = (
    matches_cleaned
    .groupby('AwayTeam')['FTHome']
    .apply(lambda s: s.shift(1).rolling(window=20, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)


In [18]:
matches_features = matches_cleaned[['HomeElo', 'AwayElo', 
    'Form3Home', 'Form5Home', 'Form3Away', 'Form5Away', 
    'FTResult', 
    'HomeTotalPointsRoll20', 'AwayTotalPointsRoll20',
    'HomeTeamHomePointsRolling20', 'AwayTeamAwayPointsRolling20', 
    'HomeTeamShotsRolling20', 'AwayTeamShotsRolling20', 
    'HomeTeamShotsHomeRolling20', 'AwayTeamShotsAwayRolling20', 
    'HomeTeamTargetShotsRolling20', 'AwayTeamTargetShotsRolling20', 
    'HomeTotalGoalsRolling20', 'AwayTotalGoalsRolling20', 
    'HomeGoalsHomeRolling20', 'AwayGoalsAwayRolling20', 
    'HomeConcededHomeRolling20','AwayConcededAwayRolling20', 'HomeOdds', 'DrawOdds', 'AwayOdds']]

In [19]:
matches_features

,HomeElo,AwayElo,Form3Home,Form5Home,Form3Away,Form5Away,FTResult,HomeTotalPointsRoll20,AwayTotalPointsRoll20,HomeTeamHomePointsRolling20,AwayTeamAwayPointsRolling20,HomeTeamShotsRolling20,AwayTeamShotsRolling20,HomeTeamShotsHomeRolling20,AwayTeamShotsAwayRolling20,HomeTeamTargetShotsRolling20,AwayTeamTargetShotsRolling20,HomeTotalGoalsRolling20,AwayTotalGoalsRolling20,HomeGoalsHomeRolling20,AwayGoalsAwayRolling20,HomeConcededHomeRolling20,AwayConcededAwayRolling20,HomeOdds,DrawOdds,AwayOdds
0,1608.77,1579.99,0.0,0.0,0.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.436364,0.290909,0.272727
1,1800.17,1681.36,0.0,0.0,0.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.583075,0.252094,0.164831
2,1635.61,1679.18,0.0,0.0,0.0,0.0,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.410959,0.294521,0.294521
3,1636.08,1630.02,0.0,0.0,0.0,0.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.440497,0.284192,0.275311
4,1782.55,1685.55,0.0,0.0,0.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.530864,0.265432,0.203704
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9320,1799.76,1880.42,5.0,8.0,6.0,12.0,2,1.70,1.55,1.50,1.55,11.65,16.05,13.30,13.80,4.3,5.55,1.80,1.30,1.45,1.65,1.10,1.40,0.309101,0.256753,0.434146
9321,1557.66,1985.45,1.0,2.0,4.0,8.0,2,0.30,1.90,0.35,1.90,9.10,14.15,8.80,11.90,3.0,4.55,0.70,1.65,0.80,1.80,2.55,0.90,0.100538,0.162964,0.736498
9322,1882.33,1776.20,4.0,7.0,7.0,7.0,2,2.00,1.45,2.10,1.10,13.75,9.65,15.85,9.70,4.7,3.90,2.05,1.35,2.30,0.85,1.05,1.20,0.721382,0.170040,0.108578
9323,2008.67,1820.28,1.0,7.0,7.0,9.0,1,2.05,1.80,2.55,1.55,17.95,13.40,18.05,13.15,5.8,5.00,2.25,1.60,2.35,1.50,0.85,1.30,0.679776,0.168440,0.151784


DIFFERENCE FEATURES

In [20]:
matches_features['TotalPointsRoll20_diff'] = matches_features['AwayTotalPointsRoll20'] - matches_features['HomeTotalPointsRoll20']

matches_features['HomeAwayPointsRolling20_diff'] = matches_features['AwayTeamAwayPointsRolling20'] - matches_features['HomeTeamHomePointsRolling20']

matches_features['ShotsRolling20_diff'] = matches_features['AwayTeamShotsRolling20'] - matches_features['HomeTeamShotsRolling20']

matches_features['ShotsHomeAwayRolling20_diff'] = matches_features['AwayTeamShotsAwayRolling20'] - matches_features['HomeTeamShotsHomeRolling20']

matches_features['TargetShotsRolling20_diff'] = matches_features['AwayTeamTargetShotsRolling20'] - matches_features['HomeTeamTargetShotsRolling20']

matches_features['GoalsRolling20_diff'] = matches_features['AwayTotalGoalsRolling20'] - matches_features['HomeTotalGoalsRolling20']

matches_features['GoalsHomeAwayRolling20_diff'] = matches_features['AwayGoalsAwayRolling20'] - matches_features['HomeGoalsHomeRolling20']

matches_features['ConcededHomeAwayRolling20_diff'] = matches_features['AwayConcededAwayRolling20'] - matches_features['HomeConcededHomeRolling20']


C:\Users\mikko\AppData\Local\Temp\ipykernel_24156\991253460.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  matches_features['TotalPointsRoll20_diff'] = matches_features['AwayTotalPointsRoll20'] - matches_features['HomeTotalPointsRoll20']
C:\Users\mikko\AppData\Local\Temp\ipykernel_24156\991253460.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  matches_features['HomeAwayPointsRolling20_diff'] = matches_features['AwayTeamAwayPointsRolling20'] - matches_features['HomeTeamHomePointsRolling20']
C:\Users

INTEGERS

In [21]:
int_cols = ['Form3Home', 'Form5Home', 'Form3Away', 'Form5Away', 'FTResult']

# convert specific columns to int
matches_features[int_cols] = matches_features[int_cols].astype('int64')

# convert the rest to float
matches_features.loc[:, ~matches_features.columns.isin(int_cols)] = matches_features.loc[:, ~matches_features.columns.isin(int_cols)].astype('float64')

C:\Users\mikko\AppData\Local\Temp\ipykernel_24156\442490929.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  matches_features[int_cols] = matches_features[int_cols].astype('int64')


FEATURES AND TARGET (X, y)

In [22]:
matches_features = matches_features.dropna()

X = matches_features.drop(['FTResult', 'HomeOdds', 'DrawOdds', 'AwayOdds'], axis=1)
y = matches_features['FTResult']

In [23]:
X.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9253 entries, 19 to 9324
Data columns (total 30 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   HomeElo                         9253 non-null   float64
 1   AwayElo                         9253 non-null   float64
 2   Form3Home                       9253 non-null   int64  
 3   Form5Home                       9253 non-null   int64  
 4   Form3Away                       9253 non-null   int64  
 5   Form5Away                       9253 non-null   int64  
 6   HomeTotalPointsRoll20           9253 non-null   float64
 7   AwayTotalPointsRoll20           9253 non-null   float64
 8   HomeTeamHomePointsRolling20     9253 non-null   float64
 9   AwayTeamAwayPointsRolling20     9253 non-null   float64
 10  HomeTeamShotsRolling20          9253 non-null   float64
 11  AwayTeamShotsRolling20          9253 non-null   float64
 12  HomeTeamShotsHomeRolling20      9253 n

TRAIN AND TEST DATA SPLIT

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=False
)

RANDOM FOREST TRAIN

In [25]:
rf = RandomForestClassifier(
    n_estimators=1500,
    max_depth=None,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features="sqrt",
    class_weight="balanced_subsample", 
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)

# Predict class + probabilities
pred_rf = rf.predict(X_test)
proba_rf = rf.predict_proba(X_test)

print("Accuracy:", accuracy_score(y_test, pred_rf))
print("LogLoss:", log_loss(y_test, proba_rf))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, pred_rf))
print("\nClassification Report:\n", classification_report(y_test, pred_rf, digits=3))

Accuracy: 0.5180983252296056
LogLoss: 0.9922140553144067

Confusion Matrix:
 [[534  72 193]
 [193  45 189]
 [178  67 380]]

Classification Report:
               precision    recall  f1-score   support

           0      0.590     0.668     0.627       799
           1      0.245     0.105     0.147       427
           2      0.499     0.608     0.548       625

    accuracy                          0.518      1851
   macro avg      0.444     0.461     0.441      1851
weighted avg      0.480     0.518     0.490      1851



In [26]:
imp = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(imp.head(25))

TargetShotsRolling20_diff         0.051744
ShotsRolling20_diff               0.051386
AwayElo                           0.051300
HomeElo                           0.048247
ShotsHomeAwayRolling20_diff       0.046555
HomeAwayPointsRolling20_diff      0.044198
TotalPointsRoll20_diff            0.041264
GoalsHomeAwayRolling20_diff       0.038076
AwayTeamShotsRolling20            0.036753
GoalsRolling20_diff               0.036711
AwayTeamShotsAwayRolling20        0.036642
HomeTeamShotsHomeRolling20        0.036101
HomeTeamShotsRolling20            0.034982
ConcededHomeAwayRolling20_diff    0.034312
HomeTeamTargetShotsRolling20      0.034026
AwayTeamTargetShotsRolling20      0.032355
HomeGoalsHomeRolling20            0.029267
HomeConcededHomeRolling20         0.028351
AwayTotalPointsRoll20             0.027963
AwayTeamAwayPointsRolling20       0.027650
HomeTeamHomePointsRolling20       0.027443
AwayConcededAwayRolling20         0.027288
HomeTotalPointsRoll20             0.026820
AwayTotalGo

In [27]:
X_rf_last = X_test.iloc[[-1]]   # double brackets keep it 2D

pred_rf_last = rf.predict(X_rf_last)
proba_rf_last = rf.predict_proba(X_rf_last)

print("Predicted class:", pred_rf_last)
print("Probabilities:", proba_rf_last)

Predicted class: [2]
Probabilities: [[0.25341822 0.26727161 0.47931017]]


In [ ]:
pred_y_rf = rf.predict(X_test)
proba_rf = rf.predict_proba(X_test)
proba_rf_df = pd.DataFrame(proba_rf, columns=['pred_rf_H', 'pred_rf_D', 'pred_rf_A'])
proba_rf_df

,pred_rf_H,pred_rf_D,pred_rf_A
0,0.167714,0.231159,0.601127
1,0.392062,0.345427,0.262512
2,0.261085,0.390630,0.348285
3,0.469652,0.308349,0.221999
4,0.476618,0.316039,0.207343
...,...,...,...
1846,0.184994,0.312077,0.502929
1847,0.124843,0.147624,0.727533
1848,0.642881,0.228213,0.128906
1849,0.648316,0.202799,0.148885


In [29]:
Features_last1851 = matches_features.iloc[-len(proba_rf_df):][['HomeOdds','DrawOdds','AwayOdds']].reset_index(drop=True)

proba_rf_df = pd.concat([proba_rf_df.reset_index(drop=True), Features_last1851], axis=1)

In [30]:
proba_rf_df['total_prob'] = proba_rf_df['HomeOdds'] + proba_rf_df['DrawOdds'] + proba_rf_df['AwayOdds']
proba_rf_df

,pred_rf_H,pred_rf_D,pred_rf_A,HomeOdds,DrawOdds,AwayOdds,total_prob
0,0.167714,0.231159,0.601127,0.200669,0.235097,0.564234,1.0
1,0.392062,0.345427,0.262512,0.454605,0.288920,0.256475,1.0
2,0.261085,0.390630,0.348285,0.357701,0.294966,0.347333,1.0
3,0.469652,0.308349,0.221999,0.591714,0.228777,0.179509,1.0
4,0.476618,0.316039,0.207343,0.507399,0.251043,0.241558,1.0
...,...,...,...,...,...,...,...
1846,0.184994,0.312077,0.502929,0.309101,0.256753,0.434146,1.0
1847,0.124843,0.147624,0.727533,0.100538,0.162964,0.736498,1.0
1848,0.642881,0.228213,0.128906,0.721382,0.170040,0.108578,1.0
1849,0.648316,0.202799,0.148885,0.679776,0.168440,0.151784,1.0


In [31]:
proba_rf_rng = rf.predict_proba(X_test)

rng = np.random.default_rng(42)

pred_rf_rng = np.array([
    rng.choice(len(p), p=p) for p in proba_rf_rng
])

In [32]:
print("Accuracy:", accuracy_score(y_test, pred_rf_rng))
print("LogLoss:", log_loss(y_test, proba_rf_rng))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, pred_rf_rng))
print("\nClassification Report:\n", classification_report(y_test, pred_rf_rng, digits=3))

Accuracy: 0.4100486223662885
LogLoss: 0.9922140553144067

Confusion Matrix:
 [[360 218 221]
 [156 130 141]
 [199 157 269]]

Classification Report:
               precision    recall  f1-score   support

           0      0.503     0.451     0.476       799
           1      0.257     0.304     0.279       427
           2      0.426     0.430     0.428       625

    accuracy                          0.410      1851
   macro avg      0.396     0.395     0.394      1851
weighted avg      0.421     0.410     0.414      1851



In [33]:

SEED = 42

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 400, 2500),
        "max_depth": trial.suggest_int("max_depth", 4, 60),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 40),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 25),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced", "balanced_subsample"]),
        "n_jobs": -1,
        "random_state": SEED,
    }

    model_rf = RandomForestClassifier(**params)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    # Optimize for probability quality (lower log loss is better)
    scores = cross_val_score(
        model_rf,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_log_loss",
        n_jobs=-1
    )

    return float(np.mean(scores))  # maximize neg_log_loss

# Study
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED)
)

study.optimize(objective, n_trials=60, show_progress_bar=True)

print("\nBest CV neg_log_loss:", study.best_value)
print("Best params:\n", study.best_params)

# Train final model with best params
rf_opt = RandomForestClassifier(
    **study.best_params,
    n_jobs=-1,
    random_state=SEED
)
rf_opt.fit(X_train, y_train)

[I 2026-03-10 20:28:32,576] A new study created in memory with name: no-name-951071b4-b6a8-4e97-a699-10cd5bc40f21
Best trial: 0. Best value: -0.999542:   2%|▏         | 1/60 [00:09<09:43,  9.89s/it]

[I 2026-03-10 20:28:42,469] Trial 0 finished with value: -0.999542371958716 and parameters: {'n_estimators': 1186, 'max_depth': 58, 'min_samples_split': 30, 'min_samples_leaf': 15, 'max_features': 'sqrt', 'bootstrap': True, 'class_weight': 'balanced_subsample'}. Best is trial 0 with value: -0.999542371958716.


Best trial: 1. Best value: -0.997586:   3%|▎         | 2/60 [00:29<15:14, 15.76s/it]

[I 2026-03-10 20:29:02,346] Trial 1 finished with value: -0.9975862205624599 and parameters: {'n_estimators': 2148, 'max_depth': 16, 'min_samples_split': 9, 'min_samples_leaf': 5, 'max_features': 'log2', 'bootstrap': False, 'class_weight': 'balanced_subsample'}. Best is trial 1 with value: -0.9975862205624599.


Best trial: 1. Best value: -0.997586:   5%|▌         | 3/60 [01:03<22:55, 24.12s/it]

...

Best trial: 54. Best value: -0.974275:  98%|█████████▊| 59/60 [22:44<00:10, 10.78s/it]

[I 2026-03-10 20:51:16,783] Trial 58 finished with value: -1.006837881185493 and parameters: {'n_estimators': 1945, 'max_depth': 6, 'min_samples_split': 14, 'min_samples_leaf': 16, 'max_features': 'sqrt', 'bootstrap': False, 'class_weight': 'balanced'}. Best is trial 54 with value: -0.9742748203256276.


Best trial: 54. Best value: -0.974275: 100%|██████████| 60/60 [22:49<00:00, 22.82s/it]


[I 2026-03-10 20:51:22,019] Trial 59 finished with value: -0.975799214452505 and parameters: {'n_estimators': 955, 'max_depth': 10, 'min_samples_split': 18, 'min_samples_leaf': 19, 'max_features': 'sqrt', 'bootstrap': True, 'class_weight': None}. Best is trial 54 with value: -0.9742748203256276.

Best CV neg_log_loss: -0.9742748203256276
Best params:
 {'n_estimators': 2277, 'max_depth': 6, 'min_samples_split': 17, 'min_samples_leaf': 21, 'max_features': 'sqrt', 'bootstrap': True, 'class_weight': None}


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",2277
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",6
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",17
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",21
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [34]:
# Evaluate on test set
pred_rf_opt = rf_opt.predict(X_test)
proba_rf_opt = rf_opt.predict_proba(X_test)

proba_rf_opt_df = pd.DataFrame(proba_rf_opt, columns=['pred_rf_opt_H', 'pred_rf_opt_D', 'pred_rf_opt_A'])

In [36]:
Features_last1851 = matches_features.iloc[-len(proba_rf_df):][['HomeOdds','DrawOdds','AwayOdds']].reset_index(drop=True)

proba_rf_opt_df = pd.concat([proba_rf_opt_df.reset_index(drop=True), Features_last1851], axis=1)

In [37]:
proba_rf_opt_df

,pred_rf_opt_H,pred_rf_opt_D,pred_rf_opt_A,HomeOdds,DrawOdds,AwayOdds
0,0.232151,0.237537,0.530312,0.200669,0.235097,0.564234
1,0.477574,0.283111,0.239315,0.454605,0.288920,0.256475
2,0.378812,0.300436,0.320751,0.357701,0.294966,0.347333
3,0.534428,0.265993,0.199579,0.591714,0.228777,0.179509
4,0.569641,0.258686,0.171673,0.507399,0.251043,0.241558
...,...,...,...,...,...,...
1846,0.270574,0.270043,0.459383,0.309101,0.256753,0.434146
1847,0.143190,0.134263,0.722547,0.100538,0.162964,0.736498
1848,0.710732,0.184115,0.105153,0.721382,0.170040,0.108578
1849,0.674934,0.195237,0.129829,0.679776,0.168440,0.151784


In [38]:
print("\nTEST Accuracy:", accuracy_score(y_test, pred_rf_opt))
print("TEST LogLoss:", log_loss(y_test, proba_rf_opt))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, pred_rf_opt))
print("\nClassification Report:\n", classification_report(y_test, pred_rf_opt, digits=3))


TEST Accuracy: 0.5353862776877364
TEST LogLoss: 0.9783339444257058

Confusion Matrix:
 [[666   0 133]
 [279   0 148]
 [300   0 325]]

Classification Report:
               precision    recall  f1-score   support

           0      0.535     0.834     0.652       799
           1      0.000     0.000     0.000       427
           2      0.536     0.520     0.528       625

    accuracy                          0.535      1851
   macro avg      0.357     0.451     0.393      1851
weighted avg      0.412     0.535     0.460      1851



c:\Users\mikko\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\mikko\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\mikko\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [42]:
proba_opt_rng = rf_opt.predict_proba(X_test)

rng = np.random.default_rng(42)

pred_opt_rng = np.array([
    rng.choice(len(p), p=p) for p in proba_opt_rng
])

In [43]:
print("TEST Accuracy:", accuracy_score(y_test, pred_opt_rng))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, pred_rf_opt))
print("\nClassification Report:\n", classification_report(y_test, pred_opt_rng, digits=3))

TEST Accuracy: 0.43381955699621827

Confusion Matrix:
 [[666   0 133]
 [279   0 148]
 [300   0 325]]

Classification Report:
               precision    recall  f1-score   support

           0      0.513     0.533     0.523       799
           1      0.273     0.279     0.276       427
           2      0.441     0.413     0.426       625

    accuracy                          0.434      1851
   macro avg      0.409     0.408     0.408      1851
weighted avg      0.433     0.434     0.433      1851



In [44]:
X_rf_last = X_test.iloc[[-1]]   # double brackets keep it 2D

pred_rf_opt_last = rf_opt.predict(X_rf_last)
proba_rf_opt_last = rf_opt.predict_proba(X_rf_last)

print("Predicted class:", pred_rf_opt_last)
print("Probabilities:", proba_rf_opt_last)

Predicted class: [2]
Probabilities: [[0.31269795 0.27015091 0.41715114]]


In [33]:
model_xb = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=5000,
    learning_rate=0.1,      
    max_depth=4,            
    min_child_weight=5,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    tree_method="hist",
    eval_metric="mlogloss",
    early_stopping_rounds=200,  
    random_state=42,
)

model_xb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=100)

#metrics
proba_xb = model_xb.predict_proba(X_test)
pred_xb = proba_xb.argmax(axis=1)

print("Accuracy:", accuracy_score(y_test, pred_xb))
print("LogLoss:", log_loss(y_test, proba_xb))

print(confusion_matrix(y_test, pred_xb))
print(classification_report(y_test, pred_xb, digits=3))

[0]	validation_0-mlogloss:1.06005
[100]	validation_0-mlogloss:0.99728
[200]	validation_0-mlogloss:1.01779
[230]	validation_0-mlogloss:1.02362
Accuracy: 0.528372591006424
LogLoss: 0.9857930185095406
[[677   3 129]
 [281   1 148]
 [314   6 309]]
              precision    recall  f1-score   support

           0      0.532     0.837     0.651       809
           1      0.100     0.002     0.005       430
           2      0.527     0.491     0.509       629

    accuracy                          0.528      1868
   macro avg      0.387     0.443     0.388      1868
weighted avg      0.431     0.528     0.454      1868



In [34]:
X_xb_w_last = X_test.iloc[[-1]]   # double brackets keep it 2D

pred_xb_last = model_xb.predict(X_xb_w_last)
proba_xb_last = model_xb.predict_proba(X_xb_w_last)

print("Predicted class:", pred_xb_last)
print("Probabilities:", proba_xb_last)

Predicted class: [2]
Probabilities: [[0.3042779  0.23125075 0.4644713 ]]


In [35]:


SEED = 42
N_CLASSES = 3

# convert y if DataFrame with one column
if hasattr(y_train, "shape") and len(y_train.shape) == 2:
    y_train = y_train.iloc[:,0]
    y_test = y_test.iloc[:,0]

def objective(trial):

    params_xgbo = {
        "objective": "multi:softprob",
        "num_class": N_CLASSES,
        "tree_method": "hist",
        "eval_metric": "mlogloss",
        "random_state": SEED,
        "n_jobs": -1,

        # hyperparameters to tune
        "n_estimators": trial.suggest_int("n_estimators", 1000, 12000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 10),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 50.0, log=True),

        "gamma": trial.suggest_float("gamma", 0.0, 10.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 5.0, log=True),

        # early stopping goes HERE for xgboost >=3
        "early_stopping_rounds": 200,
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    losses = []

    for tr_idx, va_idx in cv.split(X_train, y_train):

        X_tr = X_train.iloc[tr_idx]
        X_va = X_train.iloc[va_idx]
        y_tr = y_train.iloc[tr_idx]
        y_va = y_train.iloc[va_idx]

        model_xgb_o = xgb.XGBClassifier(**params_xgbo)

        model_xgb_o.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False
        )

        proba_xgbo = model_xgb_o.predict_proba(X_va)
        losses.append(log_loss(y_va, proba_xgbo, labels=np.arange(N_CLASSES)))

    return -np.mean(losses)


study_xgb = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED)
)

study_xgb.optimize(objective, n_trials=60, show_progress_bar=True)

print("\nBest CV neg_log_loss:", study_xgb.best_value)
print("Best params:\n", study_xgb.best_params)


# ---- train final optimized model ----

best_params_xgb = {
    **study_xgb.best_params,
    "objective": "multi:softprob",
    "num_class": N_CLASSES,
    "tree_method": "hist",
    "eval_metric": "mlogloss",
    "random_state": SEED,
    "n_jobs": -1,
    "early_stopping_rounds": 200,
}

model_xb_opt = xgb.XGBClassifier(**best_params_xgb)

model_xb_opt.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=100
)


[I 2026-03-09 21:14:47,834] A new study created in memory with name: no-name-b61601ba-99aa-4ea0-8a73-132127080ac1
Best trial: 0. Best value: -0.976339:   2%|▏         | 1/60 [00:04<04:54,  4.99s/it]

[I 2026-03-09 21:14:52,819] Trial 0 finished with value: -0.9763387531703787 and parameters: {'n_estimators': 5120, 'learning_rate': 0.2536999076681772, 'max_depth': 8, 'min_child_weight': 12.374511199743695, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.5779972601681014, 'reg_lambda': 0.0018747059221802516, 'gamma': 8.661761457749352, 'reg_alpha': 0.0016946556203947059}. Best is trial 0 with value: -0.9763387531703787.


Best trial: 1. Best value: -0.97428:   3%|▎         | 2/60 [00:13<06:52,  7.12s/it] 

[I 2026-03-09 21:15:01,428] Trial 1 finished with value: -0.9742797887563178 and parameters: {'n_estimators': 8789, 'learning_rate': 0.010725209743171996, 'max_depth': 10, 'min_child_weight': 16.816410175208013, 'subsample': 0.6061695553391381, 'colsample_bytree': 0.5909124836035503, 'reg_lambda': 0.0072746531356694235, 'gamma': 3.0424224295953772, 'reg_alpha': 0.0003671474011048667}. Best is trial 1 with value: -0.9742797887563178.


Best trial: 1. Best value: -0.97428:   5%|▌         | 3/60 [00:24<08:24,  8.85s/it]

...

[I 2026-03-09 21:21:19,446] Trial 58 finished with value: -0.9741757650094727 and parameters: {'n_estimators': 7727, 'learning_rate': 0.057245502785704006, 'max_depth': 8, 'min_child_weight': 10.992491979986042, 'subsample': 0.6882232331299647, 'colsample_bytree': 0.6628337668338861, 'reg_lambda': 0.005174320312247845, 'gamma': 3.7479576797913885, 'reg_alpha': 6.747491278803308e-06}. Best is trial 41 with value: -0.9732155657230992.


Best trial: 59. Best value: -0.973063: 100%|██████████| 60/60 [06:38<00:00,  6.64s/it]

[I 2026-03-09 21:21:26,419] Trial 59 finished with value: -0.9730629466290394 and parameters: {'n_estimators': 4332, 'learning_rate': 0.01789269576971008, 'max_depth': 10, 'min_child_weight': 4.825799127549914, 'subsample': 0.5300916575240049, 'colsample_bytree': 0.5883615110150762, 'reg_lambda': 0.02516502304244902, 'gamma': 4.614151718031739, 'reg_alpha': 2.5605700319869678e-06}. Best is trial 59 with value: -0.9730629466290394.

Best CV neg_log_loss: -0.9730629466290394
Best params:
 {'n_estimators': 4332, 'learning_rate': 0.01789269576971008, 'max_depth': 10, 'min_child_weight': 4.825799127549914, 'subsample': 0.5300916575240049, 'colsample_bytree': 0.5883615110150762, 'reg_lambda': 0.02516502304244902, 'gamma': 4.614151718031739, 'reg_alpha': 2.5605700319869678e-06}
[0]	validation_0-mlogloss:1.07076


[100]	validation_0-mlogloss:0.98523
[200]	validation_0-mlogloss:0.97929
[300]	validation_0-mlogloss:0.97921
[400]	validation_0-mlogloss:0.97936
[441]	validation_0-mlogloss:0.97930


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.5883615110150762
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",200
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import lo

In [36]:
# metrics
proba_xb_o = model_xb_opt.predict_proba(X_test)
pred_xb_o = proba_xb_o.argmax(axis=1)

print("\nAccuracy:", accuracy_score(y_test, pred_xb_o))
print("LogLoss:", log_loss(y_test, proba_xb_o))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, pred_xb_o))
print("\nClassification Report:\n", classification_report(y_test, pred_xb_o, digits=3))


Accuracy: 0.5310492505353319
LogLoss: 0.9789725151040787

Confusion Matrix:
 [[663   0 146]
 [283   0 147]
 [300   0 329]]

Classification Report:
               precision    recall  f1-score   support

           0      0.532     0.820     0.645       809
           1      0.000     0.000     0.000       430
           2      0.529     0.523     0.526       629

    accuracy                          0.531      1868
   macro avg      0.354     0.448     0.390      1868
weighted avg      0.409     0.531     0.457      1868



c:\Users\mikko\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\mikko\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\mikko\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [37]:
proba_xgbo_opt_rng = model_xb_opt.predict_proba(X_test)

rng_xgbo = np.random.default_rng(42)

pred_xgbo_opt_rng = np.array([
    rng.choice(len(p), p=p) for p in proba_xgbo_opt_rng
])

In [38]:
print("TEST Accuracy:", accuracy_score(y_test, pred_xgbo_opt_rng))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, pred_xgbo_opt_rng))
print("\nClassification Report:\n", classification_report(y_test, pred_xgbo_opt_rng, digits=3))

TEST Accuracy: 0.45610278372591007

Confusion Matrix:
 [[459 168 182]
 [185 123 122]
 [226 133 270]]

Classification Report:
               precision    recall  f1-score   support

           0      0.528     0.567     0.547       809
           1      0.290     0.286     0.288       430
           2      0.470     0.429     0.449       629

    accuracy                          0.456      1868
   macro avg      0.429     0.428     0.428      1868
weighted avg      0.454     0.456     0.454      1868



In [39]:
X_xb_o_last = X_test.iloc[[-1]]   # double brackets keep it 2D

pred_xb_o_last = model_xb_opt.predict(X_xb_o_last)
proba_xb_o_last = model_xb_opt.predict_proba(X_xb_o_last)

print("Predicted class:", pred_xb_o_last)
print("Probabilities:", proba_xb_o_last)

Predicted class: [2]
Probabilities: [[0.304218   0.25660953 0.43917248]]
